In [4]:
import os

In [5]:
%pwd

'/Users/nihal/Downloads/End-End-Kidney-classification-using-MLflow-and-DVC/research'

In [6]:
os.chdir("/Users/nihal/Downloads/End-End-Kidney-classification-using-MLflow-and-DVC")

In [7]:
%pwd

'/Users/nihal/Downloads/End-End-Kidney-classification-using-MLflow-and-DVC'

In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epoch: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

In [9]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [10]:
class configurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH, 
        params_filepath=PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "Chicken-fecal-images")
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epoch=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

In [11]:
import urllib.request as request
from zipfile import ZipFile
import time

In [12]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.base_model = tf.keras.models.load_model(
            self.config.updated_base_model_path)
        
    def train_valid_generator(self):
        datagenerator_kwargs = dict(rescale=1./255, validation_split=0.20)

        dataflow_kwargs = dict(target_size=self.config.params_image_size[:-1],
                                batch_size=self.config.params_batch_size,
                                interpolation="bilinear")
        
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs)
        
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset = "validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                horizontal_flip=True,
                vertical_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                rotation_range=20,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset = "training",
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.base_model.fit(
            self.train_generator,
            steps_per_epoch=self.steps_per_epoch,
            validation_data=self.valid_generator,
            validation_steps=self.validation_steps,
            epochs=self.config.params_epoch,
        )

        self.save_model(
             path = self.config.trained_model_path,
             model = self.base_model
        )

In [14]:
try:
    config = configurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e

[2026-02-25 19:26:11,695: INFO: common]: yaml file: /Users/nihal/Downloads/End-End-Kidney-classification-using-MLflow-and-DVC/config/config.yaml loaded successfully]
[2026-02-25 19:26:11,698: INFO: common]: yaml file: /Users/nihal/Downloads/End-End-Kidney-classification-using-MLflow-and-DVC/params.yaml loaded successfully]
[2026-02-25 19:26:11,701: INFO: common]: created directory at: artifacts]
[2026-02-25 19:26:11,703: INFO: common]: created directory at: artifacts/training]
Found 78 images belonging to 2 classes.
Found 312 images belonging to 2 classes.
Epoch 1/10
19/19 [==============================] - 33s 2s/step - loss: 9.7487 - accuracy: 0.5811 - val_loss: 20.0188 - val_accuracy: 0.6094
Epoch 2/10
19/19 [==============================] - 33s 2s/step - loss: 7.2122 - accuracy: 0.6554 - val_loss: 0.7906 - val_accuracy: 0.8438
Epoch 3/10
19/19 [==============================] - 34s 2s/step - loss: 5.3560 - accuracy: 0.6824 - val_loss: 1.0952 - val_accuracy: 0.8281
Epoch 4/10
19/19

/Users/nihal/anaconda3/envs/mlops-env/lib/python3.8/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
